# Notebook 2 - Data Quality Engine
Load the master dataset from Notebook 1 and perform data quality validations.

In [ ]:
import pandas as pd
import numpy as np

# Load output from Notebook 1
master_df = pd.read_parquet("processed/master_market_data.parquet")
master_df.head()


## Rule Catalog

In [ ]:
rule_catalog = pd.DataFrame([
    ["DQ001","Date not null","Critical"],
    ["DQ002","Symbol not null","Critical"],
    ["DQ003","Duplicate (Date,Symbol)","High"],
    ["DQ004","High >= Open","High"],
    ["DQ005","High >= Close","High"],
    ["DQ006","High >= Low","High"],
    ["DQ007","Low <= Open","High"],
    ["DQ008","Low <= Close","High"],
    ["DQ009","Volume >= 0","Critical"],
    ["DQ010","Missing metadata","Medium"],
], columns=["Rule_ID","Rule_Name","Severity"])
rule_catalog


## Validation Engine

In [ ]:
exceptions=[]

def log(rule,row,column,msg,severity):
    exceptions.append({
        "Rule_ID":rule,
        "Symbol":row.get("Symbol"),
        "Date":row.get("Date"),
        "Column":column,
        "Error":msg,
        "Severity":severity
    })

# Null checks
for col in ["Date","Symbol","Open","High","Low","Close","Volume"]:
    mask=master_df[col].isna()
    for _,r in master_df[mask].iterrows():
        log("DQ001",r,col,"Null value","Critical")

# Duplicate Date+Symbol
dup=master_df.duplicated(["Date","Symbol"],keep=False)
for _,r in master_df[dup].iterrows():
    log("DQ003",r,"Date/Symbol","Duplicate key","High")

# Price relationship rules
for _,r in master_df.iterrows():
    if pd.notna(r["High"]) and pd.notna(r["Open"]) and r["High"]<r["Open"]:
        log("DQ004",r,"High","High < Open","High")
    if pd.notna(r["High"]) and pd.notna(r["Close"]) and r["High"]<r["Close"]:
        log("DQ005",r,"High","High < Close","High")
    if pd.notna(r["High"]) and pd.notna(r["Low"]) and r["High"]<r["Low"]:
        log("DQ006",r,"High","High < Low","High")
    if pd.notna(r["Low"]) and pd.notna(r["Open"]) and r["Low"]>r["Open"]:
        log("DQ007",r,"Low","Low > Open","High")
    if pd.notna(r["Low"]) and pd.notna(r["Close"]) and r["Low"]>r["Close"]:
        log("DQ008",r,"Low","Low > Close","High")
    if pd.notna(r["Volume"]) and r["Volume"]<0:
        log("DQ009",r,"Volume","Negative Volume","Critical")
    if pd.isna(r.get("Security Name")):
        log("DQ010",r,"Security Name","Missing metadata","Medium")

exception_df=pd.DataFrame(exceptions)
exception_df.head()


## Outlier Detection

In [ ]:
master_df["Daily_Return"]=master_df.groupby("Symbol")["Close"].pct_change()
q1=master_df["Daily_Return"].quantile(0.25)
q3=master_df["Daily_Return"].quantile(0.75)
iqr=q3-q1
lower=q1-1.5*iqr
upper=q3+1.5*iqr
outliers=master_df[(master_df["Daily_Return"]<lower)|(master_df["Daily_Return"]>upper)]
print("Outliers:",len(outliers))


## Data Quality Score

In [ ]:
total=len(master_df)
failed=len(exception_df)
score=((total-failed)/total)*100 if total else 0
audit=pd.DataFrame({
    "Metric":["Total Records","Exceptions","Quality Score"],
    "Value":[total,failed,round(score,2)]
})
audit


## Save Outputs

In [ ]:
exception_df.to_csv("processed/exception_table.csv",index=False)
rule_catalog.to_csv("processed/rule_catalog.csv",index=False)
audit.to_csv("processed/audit_report.csv",index=False)
print("Files saved.")
